# OpenDrumScribe Stage 2: AI MIDI Assembly Line (Omnizart) 🏭
上传你提取的纯鼓声 `drums.wav`。自动对底鼓、军鼓、踩镲进行语义分类，输出标准 `.mid` 鼓谱。

**关键步骤：** 运行前请务必点击顶部菜单栏 `代码执行程序` > `更改附加到运行时的类型` > 选择 `T4 GPU`。

In [ ]:
# 1. Initialize AI Model (暴露安装日志以便排错，请耐心等待几分钟)
!sudo apt-get update
!sudo apt-get install python3.8 python3.8-distutils python3.8-dev -y
!curl -s https://bootstrap.pypa.io/get-pip.py -o get-pip.py
!python3.8 get-pip.py

# 核心依赖安装
!python3.8 -m pip install numpy Cython
!sudo apt-get install libsndfile-dev fluidsynth ffmpeg -y
!python3.8 -m pip install git+https://github.com/Music-and-Culture-Technology-Lab/omnizart.git

# 【核心修复】绕过 Linux 环境变量缺失，直接从 Python 底层注入参数唤醒模型
!python3.8 -c "import sys; from omnizart.cli.cli import main; sys.argv=['omnizart', 'download-checkpoints']; main()"
print("\n--- AI Model Ready! / 模型准备就绪，可以开始下一模块！ ---")

In [ ]:
# 2. Upload 'drums.wav' (上传纯净鼓声音频)
from google.colab import files
import os

if os.path.exists('drums_input.wav'):
    os.remove('drums_input.wav')
    
uploaded = files.upload()
original_file = next(iter(uploaded))
os.rename(original_file, 'drums_input.wav')
print("\n--- File ready for transcription! / 文件上传成功！ ---")

In [ ]:
# 3. Run Semantic Classification & MIDI Generation (执行自动分类流水线)
# 【核心修复】同样采用底层注入法，避开可执行文件路径丢失问题
!python3.8 -c "import sys; from omnizart.cli.cli import main; sys.argv=['omnizart', 'drum', 'transcribe', 'drums_input.wav']; main()"

In [ ]:
# 4. Download Standard MIDI (下载完美映射的鼓谱 MIDI)
if os.path.exists('drums_input.mid'):
    print("Success! Downloading MIDI... / 转换成功！正在下载标准 MIDI 文件...")
    files.download('drums_input.mid')
else:
    print("Error: MIDI not generated. Please check the logs. / 错误：未能生成 MIDI，请检查上方报错信息。")